# Init

In [ ]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd
import boto3
import re
import sys

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm


def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
SAVE_DIR = DATA_DIR / 'cleaned'
DEV_MODE = True

process_state_file = DATA_DIR / "process_state.json"
process_state = tgm.load(process_state_file)
print(len(process_state))

logger = tgl.initiate_logger('logger', SAVE_DIR / 'cleaned.log')

241


# Cleaning

## * select entities to process

In [ ]:
countries_cleaned = [c for c, val in process_state.items() if (val['clean']['status'] == 'ok')]
print('countries cleaned: ', len(countries_cleaned))
countries_to_clean = [c for c, val in process_state.items() if (val['scrape']['status'] == 'ok') and val['clean']['status'] == 'pending']
print('countries to clean: ', len(countries_to_clean))

countries cleaned:  83
countries to clean:  15


In [79]:
countries_to_clean = ['Armenia']

## * download raw data from backblaze b2

In [71]:
session = boto3.session.Session()

s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

In [ ]:
downloaded_count = 0
to_download_total = 0
raw_data_dir = Path('data/raw/osm countries queries')
list_obj_response = s3.list_objects_v2(Bucket=os.environ["B2_BUCKET_NAME"], Prefix=raw_data_dir.as_posix())
files_list = [(obj['Key']) for obj in list_obj_response['Contents']]
logger.info(f"Total files found for bucket in {raw_data_dir}: {len(files_list)}")

logger.info(f"* Downloading data from backbkaze: {len(countries_to_clean)}")
# load data from b2 bucket for countries to process
for count, country in enumerate(countries_to_clean):
    country_files = [str(file) for file in files_list if re.match(rf"{raw_data_dir.as_posix()}/{country}/.+\.json", file)]
    to_download_total += len(country_files)
    logger.info(f"* Country {country} ({count}/{len(countries_to_clean)}) files found: {len(country_files)}")
    for file in country_files:
        save_file = ROOT / raw_data_dir / country / os.path.basename(file)
        if save_file.exists():
            logger.info(f"  * Skip existing file {save_file}")
            continue
        
        os.makedirs(save_file.parent, exist_ok=True)
        try:
            s3.download_file(os.environ["B2_BUCKET_NAME"], file, str(save_file))
            logger.info(f"  * File '{file}' downloaded successfully to '{save_file}'")
            downloaded_count += 1
        except Exception as e:
            logger.error(f"  * Error downloading file '{file}': {e}")

logger.info(f"* Number of downloaded files: {downloaded_count}/{to_download_total}")

[INFO] : Total files found for bucket in data\raw\osm countries queries: 18
[INFO] : * files found for Armenia: 4
[INFO] :   * File 'data/raw/osm countries queries/Armenia/lvl_2_chunk_0_rawOSMRes.json' downloaded successfully to 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_2_chunk_0_rawOSMRes.json'
[INFO] :   * File 'data/raw/osm countries queries/Armenia/lvl_4_chunk_0_rawOSMRes.json' downloaded successfully to 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_4_chunk_0_rawOSMRes.json'
[INFO] :   * File 'data/raw/osm countries queries/Armenia/lvl_6_chunk_0_rawOSMRes.json' downloaded successfully to 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\raw\osm countries queries\Armenia\lvl_6_chunk_0_rawOSMRes.json'
[INFO] :   * File 'data/raw/osm countries queries/Armenia/lvl_8_chunk_0_rawOSMRes.json' downlo

## * load data for countries to clean

In [81]:
country_raw_dirs = [f for f in (DATA_DIR / 'raw/osm countries queries').glob('*') if f.is_dir()]
print(f"* Number of all raw directories found: {len(country_raw_dirs)}")

* Number of all raw directories found: 98


In [82]:
to_clean_by_cntr = {}
print(f"* Loading raw data only for countries to clean: {len(countries_to_clean)}")
# for chunks and non chunk files
for country in countries_to_clean:
    country_dir = DATA_DIR / 'raw/osm countries queries' / country
    if not country_dir.exists():
        continue
    files_elements = [tgm.load(f)['elements'] for f in country_dir.glob('*.json')]
    elements = [ele for list in files_elements for ele in list]
    to_clean_by_cntr[country] = elements

print(f"* Number of countries with raw data loaded: {len(to_clean_by_cntr)}")
print(f"* Number of countries to clean without raw data: {tgm.complement(countries_to_clean, to_clean_by_cntr.keys())}")

* Loading raw data only for countries to clean: 1
* Number of countries with raw data loaded: 1
* Number of countries to clean without raw data: []


## * use sovereign countries only

In [125]:
sovereign_countries = tgm.load(DATA_DIR / 'sovereign countries.json')
print(f"Sovereign countries: {len(sovereign_countries)}")

Sovereign countries: 214


In [ ]:
cleaned_by_cntr = {k:data for k,data in to_clean_by_cntr.items() if k in sovereign_countries}
print(f"Filtered sovereign countries: {len(cleaned_by_cntr)}")

# to clean data: 1
# cleaned data: 1


## * clean countries

In [127]:
import copy
def clean_country_data(raw_by_cntr):

    cleaned_by_cntr = copy.deepcopy(raw_by_cntr)  

    for country, raw_data in cleaned_by_cntr.items():

        # convert id to string
        for ele in raw_data:
            ele['id'] = str(ele['id'])

        # add parent_id to level 4 entities
        cntr_id = list(filter(lambda ele: ele['tags']['admin_level'] == '2', raw_data))[0]['id']
        for ele in raw_data:
            if ele['tags']['admin_level'] == '4':
                ele['tags']['parent_id'] = str(cntr_id)

        # add country_name and country_id tag
        for ele in raw_data:
            ele['tags']['country_name'] = country
            ele['tags']['country_id'] = cntr_id

    return cleaned_by_cntr

In [129]:
cleaned_by_cntr = clean_country_data(cleaned_by_cntr)
print(f"clean countries: {len(cleaned_by_cntr)}")

clean countries: 1


In [ ]:
temp = [ele['tags'].get('parent_id', None) for k,v in cleaned_by_cntr.items() for ele in v]
print(f"Tally 'parent_id' for all elements: {tgm.tally([type(ele) for ele in temp])}")

Tally 'parent_id' for all elements: Counter({<class 'str'>: 26, <class 'NoneType'>: 1})
Counter({<class 'str'>: 26, <class 'NoneType'>: 1})


## * convert to dataframe

In [122]:
cleaned_by_cntr = {k:too.normalizeOSM(elems) for k,elems in cleaned_by_cntr.items()}
print(f"Cleaned converted to data frame: {len(cleaned_by_cntr)}")

Cleaned converted to data frame: 1


In [ ]:
combined = pd.concat(cleaned_by_cntr.values(), ignore_index=True)
print(f"Tally all types of values in dataframe: {tgm.tally(list(combined.map(type).stack().values))}")

All types of data in dataframe:
Counter({<class 'str'>: 244, <class 'pandas._libs.missing.NAType'>: 188})


## * save

In [ ]:
# save data by country
print(f"Finished cleanning. Total of cleaned countries: {len(cleaned_by_cntr)}")
print("Saving files to cleaned directory")
for country,df in cleaned_by_cntr.items():
    tgm.dump(str(SAVE_DIR / country / f'{country}_cleaned.pkl'), df)
print(f"Saved files to cleaned directory: {len(cleaned_by_cntr)}")

Total of cleaned countries: 11


## * Upload data to backblaze b2 and update process state

In [ ]:
countries_to_add = list(cleaned_by_cntr.keys())
print(len(countries_to_add))
print(countries_to_add)

In [ ]:
#* Upload data to backblaze b2 and update process state
logger.info("* Uploading data to backblaze b2")
config = {'root':ROOT, 's3':s3, 'logger':logger}
for country in cleaned_by_cntr.keys():
    logger.info(f"  * Uploading directory for country: {country}")
    country_save_dir = SAVE_DIR / country
    # all data in country directory will  be uploaded
    if not DEV_MODE:
        upload_response = tsm.upload_dir_files_to_backblaze(country_save_dir, config)
        process_status = upload_response['status']
        process_error = upload_response['status_type']
    # override process task state with upload response
    logger.info(f"  * Updating {country} in process state: (clean, ok)")
    tsm.update_process_state(process_state, country, 'clean', process_status=process_status, process_error=process_error)
    tgm.dump(process_state_file, process_state)
    if not DEV_MODE:
        tsm.commit_file(process_state_file, f"Update process state for {country}: (scrape, ok)", config['logger'])

## * review

In [61]:
rawFiles = [f for f in (DATA_DIR / 'cleaned').rglob('*.pkl')]
print(len(rawFiles))

82


In [37]:
# import
rawFiles = [f for f in (DATA_DIR / 'cleaned').rglob('*.pkl')]
print(len(rawFiles))

df_by_cntr = { file.parent.name:tgm.load(str(file)) for file in rawFiles}
print(len(df_by_cntr))

82
82


In [38]:
for k,df in df_by_cntr.items():
    cntr = df[df['tags.admin_level']=='2']
    print([k, cntr['tags.name'].iloc[0], cntr['tags.name:en'].iloc[0]])

['Afghanistan', 'افغانستان', 'Afghanistan']
['Albania', 'Shqipëria', 'Albania']
['Algeria', 'Algérie ⵍⵣⵣⴰⵢⴻⵔ الجزائر', 'Algeria']
['Andorra', 'Andorra', 'Andorra']
['Angola', 'Angola', 'Angola']
['Anguilla', 'Anguilla', 'Anguilla']
['AntiguaAndBarbuda', 'Antigua and Barbuda', 'Antigua and Barbuda']
['Argentina', 'Argentina', 'Argentina']
['Australia', 'Australia', 'Australia']
['Austria', 'Österreich', 'Austria']
['Azerbaijan', 'Azərbaycan', 'Azerbaijan']
['Bahamas', 'The Bahamas', 'Bahamas']
['Bahrain', 'البحرين', 'Bahrain']
['Bangladesh', 'বাংলাদেশ', 'Bangladesh']
['Barbados', 'Barbados', 'Barbados']
['Belarus', 'Беларусь', 'Belarus']
['Belgium', 'België / Belgique / Belgien', 'Belgium']
['Belize', 'Belize', 'Belize']
['Benin', 'Bénin', 'Benin']
['Bermuda', 'Bermuda', 'Bermuda']
['Bhutan', 'འབྲུག་ཡུལ།', 'Bhutan']
['Bolivia', 'Bolivia', 'Bolivia']
['BosniaAndHerzegovina', 'Bosna i Hercegovina / Босна и Херцеговина', 'Bosnia and Herzegovina']
['Botswana', 'Botswana', 'Botswana']
['Braz

In [40]:
resume = {}
for country, df in df_by_cntr.items():
    resume[country] = {}
    resume[country]['lvl1'] = len(df[df['tags.admin_level'] == '4'])
    resume[country]['lvl2'] = len(df[df['tags.admin_level'] == '6'])
    resume[country]['lvl3'] = len(df[df['tags.admin_level'] == '8'])
    resume[country]['total'] = len(df[df['tags.admin_level'] != '2'])

resume = pd.DataFrame(resume).T
resume.peek()

,lvl1,lvl2,lvl3,total
Afghanistan,34,0,0,34
Albania,3,12,374,389
Algeria,58,547,1542,2147
Andorra,0,0,0,0
Angola,19,29,15,63
Anguilla,0,0,0,0
AntiguaAndBarbuda,3,8,0,11
Argentina,25,661,524,1210
Australia,15,600,0,615
Austria,9,93,2075,2177
